# MiniCPM-o 4.5 Multimodal Model with OpenVINO

[MiniCPM-o 4.5](https://huggingface.co/openbmb/MiniCPM-o-4_5) is the latest end-to-end omnimodal model in the MiniCPM-o series. With **9B parameters** built on **SigLip2 + Whisper-medium + CosyVoice2 + Qwen3-8B**, it achieves Gemini 2.5 Flash level performance on vision-language benchmarks while supporting full-duplex multimodal live streaming.

<p align="center">
    <img src="https://raw.githubusercontent.com/OpenBMB/MiniCPM-o/main/assets/minicpm-o-45-framework.png" width="100%"/>
</p>

**Key capabilities:**
- 🔥 Leading visual understanding (77.6 on OpenCompass, surpassing GPT-4o)
- 🎙 Bilingual (EN/ZH) real-time speech conversation with voice cloning
- 🎬 Full-duplex multimodal live streaming (see, listen, speak simultaneously)
- 💪 State-of-the-art OCR and document parsing

In this notebook, we convert all 15 sub-models to OpenVINO IR format with INT4/INT8 weight compression, run inference examples (aligned with the [official model examples](https://huggingface.co/openbmb/MiniCPM-o-4_5)), and build an interactive Gradio demo.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert and Quantize Model](#Convert-and-Quantize-Model)
- [Select Inference Device](#Select-Inference-Device)
- [Initialize Model](#Initialize-Model)
- [Visual Understanding](#Visual-Understanding)
    - [Chat with Single Image](#Chat-with-Single-Image)
    - [Chat with Multiple Images](#Chat-with-Multiple-Images)
    - [Multi-turn Conversation](#Multi-turn-Conversation)
- [Speech and Audio Mode](#Speech-and-Audio-Mode)
    - [Audio Understanding (ASR)](#Audio-Understanding-ASR)
    - [More Audio Tasks](#More-Audio-Tasks)
- [Half-Duplex Omni Mode](#Half-Duplex-Omni-Mode)
    - [Chat Inference](#Chat-Inference)
    - [Streaming Inference](#Streaming-Inference)
- [Half-Duplex Realtime Speech Conversation](#Half-Duplex-Realtime-Speech-Conversation)
- [Duplex Omni Mode](#Duplex-Omni-Mode)
- [Interactive Demo](#Interactive-Demo)

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

Install required packages and download helper files.

In [ ]:
import requests
from pathlib import Path
import sys

notebook_utils_path = Path("notebook_utils.py")
if not notebook_utils_path.exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    notebook_utils_path.write_text(r.text)

In [ ]:
%pip install -q "transformers==4.51.0" "openvino>=2025.1" "openvino-tokenizers>=2025.1" "nncf>=2.16" \
    "torch>=2.3" "torchaudio" "Pillow" "librosa" "soundfile" "gradio>=4.19" "accelerate" \
    "minicpmo-utils>=1.0.5" --extra-index-url https://download.pytorch.org/whl/cpu

## Convert and Quantize Model
[back to top ⬆️](#Table-of-contents:)

MiniCPM-o 4.5 consists of **15 interconnected sub-models**. We convert each to OpenVINO IR format:

| Sub-model | Role | Quantization |
|-----------|------|:---:|
| **LLM Embedding** | Token embeddings (Qwen3-8B) | — |
| **LLM Language Model** | Main language model with stateful KV cache | **INT4** |
| **Vision Model** | SigLip2 vision encoder | **INT8** |
| **Resampler** | Vision→LLM feature projector | — |
| **Audio Encoder** | Whisper-medium speech encoder | — |
| **Audio Projection** | Audio→LLM feature projector | — |
| **TTS Text Embedding** | TTS decoder token embeddings | — |
| **TTS Language Model** | LLaMA decoder for speech token generation | — |
| **TTS Projector SPK / Semantic** | Speaker & semantic projectors | — |
| **TTS Code Embedding / Head** | Audio code tokens embedding & prediction | — |
| **Flow Embeddings + Estimator** | CosyVoice2 flow-matching for mel generation | — |
| **HiFT** | Neural vocoder (mel→waveform) | — |

- **LLM**: INT4 symmetric quantization (`group_size=64`, `ratio=1.0`) via NNCF for best quality/size tradeoff
- **Vision**: INT8 symmetric quantization for moderate compression
- **Other models**: FP16/FP32 (small enough that quantization is unnecessary)

In [ ]:
from pathlib import Path

# Path to the original model (downloaded from HuggingFace/ModelScope)
original_model_path = Path(r"D:\MiniCPM-O-4_5\MiniCPM-o-4_5")

# Output path for converted OpenVINO models
ov_model_path = Path("MiniCPM-o-4_5-OV")

print(f"Original model: {original_model_path}")
print(f"OpenVINO output: {ov_model_path}")

In [ ]:
from minicpm_o_4_5_helper import convert_minicpmo_model

if not (ov_model_path / "openvino_llm_embedding_model.xml").exists():
    # Quantization config:
    # - "vision": INT8 for vision encoder (moderate compression)
    # - "llm": INT4_SYM with group_size=64, ratio=1.0 for LLM (best quality/size)
    # - "text": No quantization for other text/audio/TTS models
    quantization_config = {
        "vision": {
            "mode": "int8_sym",
        },
        "llm": {
            "mode": "int4_sym",
            "group_size": 64,
            "ratio": 1.0,
        },
        "text": None,
    }

    convert_minicpmo_model(
        model_id=str(original_model_path),
        output_dir=str(ov_model_path),
        quantization_config=quantization_config,
    )
    print("✅ All 15 sub-models converted successfully!")
else:
    print(f"✅ Converted models already exist at {ov_model_path}")

## Select Inference Device
[back to top ⬆️](#Table-of-contents:)

Select the OpenVINO device for inferencing. GPU is recommended for the best performance when available.

In [ ]:
from notebook_utils import device_widget

# Main device for LLM, Vision, Audio
device = device_widget("CPU", exclude=["NPU"])
device

In [ ]:
# TTS device (can be different from main device)
tts_device = device_widget("CPU", exclude=["NPU"])
tts_device

## Initialize Model
[back to top ⬆️](#Table-of-contents:)

Load all 15 OpenVINO sub-models and initialize the inference pipeline.

In [ ]:
from minicpm_o_4_5_helper import OVMiniCPMO

ov_model = OVMiniCPMO(
    model_path=str(ov_model_path),
    device=device.value,
    tts_device=tts_device.value,
)

## Visual Understanding
[back to top ⬆️](#Table-of-contents:)

MiniCPM-o 4.5 shares the same visual inference methods as MiniCPM-V-4.5.

### Chat with Single Image

Let's test the model with a single image input — a classic vision-language task.

In [ ]:
from PIL import Image
import requests
from io import BytesIO

# Download a sample image

image_url = "https://github.com/openvinotoolkit/openvino_notebooks/assets/29454499/d5fbbd1a-d484-415c-88cb-9986625b7b11"
response = requests.get(image_url, timeout=30)
image = Image.open(BytesIO(response.content)).convert("RGB")

# Resize for efficiency
image.thumbnail((512, 512))
display(image)

In [ ]:
# Chat with single image (aligned with original model README)
question = "What is in the image?"
msgs = [{"role": "user", "content": [image, question]}]

answer = ov_model.chat(
    msgs=msgs,
    use_tts_template=False,
)
print(answer)

### Chat with Multiple Images

The model can compare and reason across multiple images simultaneously.

In [ ]:
# Chat with multiple images (aligned with original model README)
from io import BytesIO

# Download two sample images
image_urls = [
    "https://github.com/openvinotoolkit/openvino_notebooks/assets/29454499/d5fbbd1a-d484-415c-88cb-9986625b7b11",
    "https://github.com/openvinotoolkit/openvino_notebooks/assets/29454499/63c01f7c-02a4-4009-b2c7-f733b4afe3b0",
]
images_multi = []
for url in image_urls:
    resp = requests.get(url, timeout=30)
    img = Image.open(BytesIO(resp.content)).convert("RGB")
    img.thumbnail((512, 512))
    images_multi.append(img)

# Reset LLM state
ov_model.llm._ov_language.reset_state()
ov_model.llm._past_length = 0

question = "Compare image 1 and image 2, tell me about the differences between them."
msgs = [{"role": "user", "content": images_multi + [question]}]

answer_multi = ov_model.chat(
    msgs=msgs,
    use_tts_template=False,
    enable_thinking=False,
)
print(answer_multi)

### Multi-turn Conversation

The model supports multi-turn conversations with context awareness across turns.

In [ ]:
# Multi-turn: follow up on the previous image
# Reset LLM state for fresh generation
ov_model.llm._ov_language.reset_state()
ov_model.llm._past_length = 0

msgs = [
    {"role": "user", "content": [image, "What is in this image?"]},
    {"role": "assistant", "content": [answer]},
    {"role": "user", "content": ["What season do you think this picture was taken in? Why?"]},
]

answer2 = ov_model.chat(
    msgs=msgs,
    use_tts_template=False,
    enable_thinking=False,
)
print(answer2)

## Speech and Audio Mode
[back to top ⬆️](#Table-of-contents:)

MiniCPM-o 4.5 can handle various audio understanding tasks. You can switch between different tasks by changing the prompt text:

| Task | Prompt |
|------|--------|
| **ASR (English)** | `Please listen to the audio snippet carefully and transcribe the content.` |
| **ASR (Chinese / AST EN→ZH)** | `请仔细听这段音频片段，并将其内容逐字记录。` |
| **Speaker Analysis** | `Based on the speaker's content, speculate on their gender, condition, age range, and health status.` |
| **General Audio Caption** | `Summarize the main content of the audio.` |
| **Sound Scene Tagging** | `Utilize one keyword to convey the audio's content or the associated scene.` |

### Audio Understanding (ASR)

In [ ]:
import librosa
import numpy as np

# Download a sample audio file (English speech)
audio_url = "https://github.com/openvinotoolkit/openvino_notebooks/assets/29454499/7a1c38ff-ccee-49e8-bce4-73ee9aa76d76"
audio_path = Path("sample_audio.wav")

if not audio_path.exists():
    r = requests.get(audio_url, timeout=30)
    audio_path.write_bytes(r.content)

# Load audio at 16kHz mono (required by MiniCPM-o)
audio_input, sr = librosa.load(str(audio_path), sr=16000, mono=True)
print(f"Audio loaded: {len(audio_input)/sr:.1f}s at {sr}Hz")

# Reset LLM state
ov_model.llm._ov_language.reset_state()
ov_model.llm._past_length = 0

# ASR task (aligned with original model README)
task_prompt = "Please listen to the audio snippet carefully and transcribe the content."
msgs = [{"role": "user", "content": [task_prompt, audio_input]}]

asr_result = ov_model.chat(
    msgs=msgs,
    do_sample=True,
    max_new_tokens=512,
    use_tts_template=True,
    temperature=0.3,
)
print("ASR Result:", asr_result)

### More Audio Tasks

Let's try additional audio understanding capabilities: Speaker Analysis and Sound Scene Tagging.

In [ ]:
# Speaker Analysis (aligned with original model README)
ov_model.llm._ov_language.reset_state()
ov_model.llm._past_length = 0

speaker_prompt = "Based on the speaker's content, speculate on their gender, condition, age range, and health status."
msgs = [{"role": "user", "content": [speaker_prompt, audio_input]}]

speaker_result = ov_model.chat(
    msgs=msgs,
    do_sample=True,
    max_new_tokens=512,
    use_tts_template=True,
    temperature=0.3,
)
print("Speaker Analysis:", speaker_result)

# Sound Scene Tagging
ov_model.llm._ov_language.reset_state()
ov_model.llm._past_length = 0

scene_prompt = "Utilize one keyword to convey the audio's content or the associated scene."
msgs = [{"role": "user", "content": [scene_prompt, audio_input]}]

scene_result = ov_model.chat(
    msgs=msgs,
    do_sample=True,
    max_new_tokens=64,
    use_tts_template=True,
    temperature=0.3,
)
print("Sound Scene Tag:", scene_result)

## Half-Duplex Omni Mode
[back to top ⬆️](#Table-of-contents:)

The **Half-Duplex Omni Mode** combines vision, audio, and text inputs into a single inference call using `omni_mode=True`. This is the primary mode for multimodal understanding tasks that involve both visual and auditory information (e.g., describing a video with sound).

We provide two inference modes aligned with the [original model documentation](https://huggingface.co/openbmb/MiniCPM-o-4_5):
- **Chat Inference** — single-pass generation with full response
- **Streaming Inference** — token-by-token generation for real-time display

### Chat Inference

In [ ]:
# Half-Duplex Omni Mode — Chat Inference
# Aligned with original model README "Half-Duplex Omni Mode > Chat Inference"
# Combines audio input with omni_mode=True for unified multimodal processing

# Reset LLM state
ov_model.llm._ov_language.reset_state()
ov_model.llm._past_length = 0

# Get system prompt for omni mode (voice cloning supported with ref_audio)
sys_msg = ov_model.get_sys_prompt(mode="omni", language="en")

# Build omni content — audio + text question
# In the original model, this can include video frames + audio segments
# extracted via get_video_frame_audio_segments(video_path)
omni_content = [audio_input, "Please describe what you hear in this audio."]
user_msg = {"role": "user", "content": omni_content}
msgs = [sys_msg, user_msg]

# Chat inference with omni mode enabled
omni_result = ov_model.chat(
    msgs=msgs,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    use_tts_template=True,
    enable_thinking=False,
    omni_mode=True,       # Required for omni inference
    max_slice_nums=1,     # Increase for HD mode with video frames
)
print("Omni Chat Result:", omni_result)

### Streaming Inference

MiniCPM-o 4.5 supports streaming text generation for real-time output. The `stream=True` parameter returns a generator that yields text chunks token-by-token as they are produced, rather than waiting for the entire response.

This is aligned with the [**Streaming Inference** section](https://huggingface.co/openbmb/MiniCPM-o-4_5#streaming-inference) in the original model README under Half-Duplex Omni Mode.

In [ ]:
# Streaming text generation (aligned with original model README "Streaming Inference")
import sys

ov_model.llm._ov_language.reset_state()
ov_model.llm._past_length = 0

question = "Write a short poem about artificial intelligence in 4 lines."
msgs = [{"role": "user", "content": [image, question]}]

# stream=True returns a generator that yields text chunks
stream_gen = ov_model.chat(
    msgs=msgs,
    max_new_tokens=256,
    stream=True,
    use_tts_template=False,
    do_sample=True,
    temperature=0.7,
)

print("Streaming output: ", end="", flush=True)
full_response = ""
for chunk in stream_gen:
    full_response += chunk
    print(chunk, end="", flush=True)
print("\n\n--- Streaming complete ---")
print(f"Total tokens: {len(full_response.split())}")

## Half-Duplex Realtime Speech Conversation
[back to top ⬆️](#Table-of-contents:)

The **Half-Duplex Realtime Speech Conversation Mode** simulates real-time voice interaction by splitting the user's audio into 1-second chunks and processing them sequentially through the streaming pipeline. The model listens to all chunks, then generates a text (and optionally audio) response.

This is aligned with the [**Half-Duplex Realtime Speech Conversation Mode** section](https://huggingface.co/openbmb/MiniCPM-o-4_5#half-duplex-realtime-speech-conversation-mode) in the original model README.

> **Note:** In our OpenVINO implementation, we use the **Duplex API** (`as_duplex()`) to achieve the same chunked audio streaming functionality. The Duplex API provides `streaming_prefill()` for feeding audio chunks and `streaming_generate()` for producing responses — functionally equivalent to the original half-duplex streaming API.

In [ ]:
# Half-Duplex Realtime Speech Conversation Mode
# Aligned with original model README "Half-Duplex Realtime Speech Conversation Mode"
# Simulates real-time speech by splitting audio into 1-second chunks

import numpy as np

# Reset LLM state for fresh generation
ov_model.llm._ov_language.reset_state()
ov_model.llm._past_length = 0

# Create duplex model for streaming
duplex_model = ov_model.as_duplex()

# System prompt for English speech conversation
system_prompt = (
    "Please assist users while maintaining a helpful and friendly style. "
    "Please answer the user's questions seriously and in a high quality. "
    "Please chat with the user in a highly human-like and oral style."
)

# Prepare duplex session (no ref_audio = default voice, text-only output)
duplex_model.prepare(
    prefix_system_prompt=system_prompt,
    generate_audio=False,
)

# Configuration (aligned with original README)
IN_SAMPLE_RATE = 16000   # Input audio sample rate
CHUNK_SAMPLES = IN_SAMPLE_RATE  # 1 second per chunk
MIN_AUDIO_SAMPLES = 16000  # Minimum audio length

# Split user audio into 1-second chunks (simulating real-time streaming)
total_samples = len(audio_input)
num_chunks = (total_samples + CHUNK_SAMPLES - 1) // CHUNK_SAMPLES
print(f"Processing {total_samples/IN_SAMPLE_RATE:.1f}s audio in {num_chunks} chunk(s)...")

for chunk_idx in range(num_chunks):
    start = chunk_idx * CHUNK_SAMPLES
    end = min((chunk_idx + 1) * CHUNK_SAMPLES, total_samples)
    chunk_audio = audio_input[start:end]

    # Pad last chunk if too short (aligned with original README)
    is_last_chunk = (chunk_idx == num_chunks - 1)
    if is_last_chunk and len(chunk_audio) < MIN_AUDIO_SAMPLES:
        chunk_audio = np.concatenate([
            chunk_audio,
            np.zeros(MIN_AUDIO_SAMPLES - len(chunk_audio), dtype=chunk_audio.dtype)
        ])

    # Step 1: Streaming prefill — feed 1-second audio chunk
    duplex_model.streaming_prefill(audio_waveform=chunk_audio)

# Step 2: Streaming generate — model produces response
result = duplex_model.streaming_generate(
    max_new_speak_tokens_per_chunk=40,
    decode_mode="sampling",
)

if not result.get("is_listen", True):
    print(f"\n🤖 AI Response: {result.get('text', '')}")
    if result.get("audio_waveform") is not None:
        print(f"🔊 Audio: {len(result['audio_waveform'])/24000:.1f}s generated")
else:
    print("👂 Model is still listening (no response generated yet)")

print("\n--- Half-Duplex Realtime Speech Conversation complete ---")

## Duplex Omni Mode
[back to top ⬆️](#Table-of-contents:)

The **Duplex Omni Mode** enables full-duplex streaming: the model can **listen and speak simultaneously**, processing multimodal input (video frames + audio chunks) in real time. This is the most advanced interaction mode, ideal for real-time conversations where the AI needs to respond while the user is still speaking.

This is aligned with the [**Duplex Omni Mode** section](https://huggingface.co/openbmb/MiniCPM-o-4_5#duplex-omni-mode) in the original model README.

The duplex pipeline works in a loop:
1. **`streaming_prefill()`** — Feed one audio chunk (+ optional video frames) to the model
2. **`streaming_generate()`** — Model decides to **listen** or **speak**, returning text + optional audio

> For a complete duplex voice interaction demo, see the **`english_tutor_demo.py`** included in this notebook folder — an AI English Speaking Coach with real-time voice streaming.

In [ ]:
# Duplex Omni Mode — Full-duplex streaming with listen/speak decisions
# Aligned with original model README "Duplex Omni Mode"
# The model processes audio chunks and decides whether to listen or speak

import numpy as np

# Reset LLM state
ov_model.llm._ov_language.reset_state()
ov_model.llm._past_length = 0

# Create duplex model (reuses existing OV models, no reloading)
duplex_model = ov_model.as_duplex()

# Prepare duplex session with system prompt
# In the original model, you can also provide ref_audio for voice cloning:
#   ref_audio, _ = librosa.load("ref_audio.wav", sr=16000, mono=True)
#   duplex_model.prepare(..., ref_audio=ref_audio, prompt_wav_path="ref_audio.wav")
duplex_model.prepare(
    prefix_system_prompt="Streaming Omni Conversation.",
    generate_audio=False,  # Set True with ref_audio for TTS output
)

# Process audio in streaming fashion (1-second chunks)
CHUNK_SAMPLES = 16000  # 1 second at 16kHz
num_chunks = (len(audio_input) + CHUNK_SAMPLES - 1) // CHUNK_SAMPLES

print(f"Duplex streaming: {num_chunks} chunks from {len(audio_input)/16000:.1f}s audio\n")

results_log = []
for chunk_idx in range(num_chunks):
    start = chunk_idx * CHUNK_SAMPLES
    end = min(start + CHUNK_SAMPLES, len(audio_input))
    chunk = audio_input[start:end]

    # Pad if needed
    if len(chunk) < CHUNK_SAMPLES:
        chunk = np.pad(chunk, (0, CHUNK_SAMPLES - len(chunk)))

    # Step 1: Streaming prefill — feed audio chunk
    # In the original model, you can also pass frame_list=[video_frame] for video
    duplex_model.streaming_prefill(
        audio_waveform=chunk,
        frame_list=None,         # Add video frames here for video conversations
        max_slice_nums=1,        # Increase for HD mode
        batch_vision_feed=False,  # Set True for faster processing
    )

    # Step 2: Streaming generate — model decides to listen or speak
    result = duplex_model.streaming_generate(
        max_new_speak_tokens_per_chunk=20,
        decode_mode="sampling",
    )

    # Log results (aligned with original model's results_log format)
    chunk_result = {
        "chunk_idx": chunk_idx,
        "is_listen": result.get("is_listen", True),
        "text": result.get("text", ""),
        "end_of_turn": result.get("end_of_turn", False),
        "audio_length": len(result["audio_waveform"]) if result.get("audio_waveform") is not None else 0,
    }
    results_log.append(chunk_result)

    # Display listen/speak decisions (aligned with original model output format)
    if result.get("is_listen", True):
        print(f"  Chunk {chunk_idx}: listen...")
    else:
        print(f"  Chunk {chunk_idx}: speak> {result.get('text', '')}")

print(f"\n--- Duplex streaming complete: {len(results_log)} chunks processed ---")
speak_chunks = [r for r in results_log if not r["is_listen"]]
print(f"Listen chunks: {len(results_log) - len(speak_chunks)}, Speak chunks: {len(speak_chunks)}")

## Interactive Demo
[back to top ⬆️](#Table-of-contents:)

Launch a Gradio-based multimodal chatbot with features inspired by the [official MiniCPM-V-CookBook demo](https://github.com/OpenSQZ/MiniCPM-V-CookBook):
- 💬 **Chat Tab** — text, image, and audio inputs with streaming output
- 🧠 **Thinking Mode** — toggle to see the model's reasoning process
- 📚 **Few-Shot Tab** — add example pairs, then generate with pattern matching
- ⏹ **Stop Button** — interrupt generation at any time
- 🎛️ **Sampling Controls** — temperature, top-p, top-k, repetition penalty

> **Duplex Voice Demo:** For a full-duplex real-time voice interaction demo (AI English Speaking Coach), run `python english_tutor_demo.py --model-path MiniCPM-o-4_5-OV` from this notebook's directory.

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_model)
try:
    demo.launch(debug=False, height=800)
except Exception:
    demo.launch(debug=False, share=True, height=800)

In [ ]:
# Cleanup
demo.close()